In [1]:
import polars as pl
from datetime import timedelta

## Читаем файлы

In [2]:
df_txns = pl.read_csv('../data/raw/transactions.csv', try_parse_dates=True)
df_clients = pl.read_csv('../data/raw/clients.csv', try_parse_dates=True)

Находим точку отсчёта считаем активность за последние 2, 3 или 6 месяцев

In [3]:
max_date = df_txns.select(pl.col('txn_date').max()).item()

date_2m = max_date - timedelta(days=60)
date_3m = max_date - timedelta(days=90)
date_6m = max_date - timedelta(days=180)

## Группировка и подсчёт

In [4]:
txns_summary = df_txns.group_by('client_id').agg([
    pl.col('txn_date').min().alias('first_txn_date'),
    pl.col('txn_date').max().alias('last_txn_date'),
    (pl.col('txn_date') >= date_2m).sum().alias('txn_cnt_2m'),
    (pl.col('txn_date') >= date_3m).sum().alias('txn_cnt_3m'),
    (pl.col('txn_date') >= date_6m).sum().alias('txn_cnt_6m')
])

## Соединяем с профилем клиента

In [5]:
result_table = (
    df_clients
    .select(['client_id', 'registration_date'])
    .join(txns_summary, on='client_id', how='left')
    .with_columns([
            pl.col('txn_cnt_2m').fill_null(0),
            pl.col('txn_cnt_3m').fill_null(0),
            pl.col('txn_cnt_6m').fill_null(0)
        ])
)


## Вывод

In [6]:
result_table

client_id,registration_date,first_txn_date,last_txn_date,txn_cnt_2m,txn_cnt_3m,txn_cnt_6m
str,date,date,date,u32,u32,u32
"""CL0000001""",2013-12-02,2024-01-02,2026-05-24,22,33,66
"""CL0000002""",2000-02-26,2024-01-02,2026-05-26,22,33,65
"""CL0000003""",2002-07-31,2024-01-01,2026-05-31,20,30,59
"""CL0000004""",2005-04-04,2024-01-01,2026-05-31,22,33,66
"""CL0000005""",2022-06-12,2024-01-07,2026-05-31,22,33,66
…,…,…,…,…,…,…
"""CL0002996""",2013-06-19,2024-01-04,2026-05-29,22,32,66
"""CL0002997""",2012-08-02,2024-01-01,2026-05-30,22,32,66
"""CL0002998""",2012-11-08,2024-01-01,2026-05-23,20,30,60
